In [2]:
import requests
from bs4 import BeautifulSoup
import os
import datetime
import time
import re
import keyring

In [5]:
# Retrieve the credentials from keyring. To set for the first time, run
# python -m keyring set nitrc.org username
username = 'yushkevich'
password = keyring.get_password('nitrc.org', username)

In [6]:
# Start a session and login to NITRC
session = requests.Session()
session.post(url='https://www.nitrc.org/account/login.php',
                  data={
                      'form_loginname': username,
                      'form_pw': password,
                      'login': '',
                      'return_to': ''})

<Response [200]>

In [7]:
def new_release(session, package_id:int, release_name: str, url_target: str, url_name: str, url_platform: str):
    payload = {
        "package_id": str(package_id),
        "release_name": release_name,
        "release_date": datetime.datetime.now().strftime('%Y-%m-%d %H:%M'),
        "number_of_volumes_1":"",
        "doi_1":"",
        "FILE_title_1":"",
        "FILE_description_1":"",
        "FILE_citation_1": "Yushkevich, P., Zhang, G., & Developer, I. (2025, Aug 30).  [.]. Washington: NITRC.",
        "FILE_creator_1": "Yushkevich, P., Zhang, G., & Developer, I.",
        "FILE_publication_year_1": "2025",
        "FILE_version_1":"",
        "FILE_licence_1": "GNU General Public License (GPL)",
        "other_processor_1":"",
        "url_line_1_exists": "true",
        "url_processor_id_1": url_platform,
        "url_location_1": url_target,
        "url_name_1": url_name,
        "number_of_volumes_1":"",
        "url_doi_1":"",
        "URL_title_1": "test01",
        "URL_description_1":"",
        "URL_citation_1": "Yushkevich, P., Zhang, G., & Developer, I. (2025, Aug 30).  [.]. Washington: NITRC.",
        "URL_creator_1": "Yushkevich, P., Zhang, G., & Developer, I.",
        "URL_publication_year_1": "2025",
        "URL_version_1":"",
        "URL_licence_1": "GNU General Public License (GPL)",
        "url_other_processor_1":"",
        "release_notes":"",
        "release_changes":"",
        "license_agreement":"",
        "submit": "Release File"
    }
    r = session.post('https://www.nitrc.org/frs/admin/qrs.php?group_id=110', data=payload)
    return r

def parse_release_id(response):
    bs = BeautifulSoup(response.content)
    return int(re.match('.*release_id=([0-9]+)&', bs.find('a', href=re.compile('.*edit.*'))['href'])[1])

def add_link_to_release(session, package_id:int, release_id:int, release_name: str, url_target: str, url_name: str, url_platform: str):
    payload = {
        "package_id": str(package_id),
        "release_name": release_name,
        "release_date": datetime.datetime.now().strftime('%Y-%m-%d %H:%M'),
        "step2": "1",
        "upload_line_1_exists": "true",
        "number_of_volumes_1":"",
        "doi_1":"",
        "FILE_title_1":"",
        "FILE_description_1":"",
        "FILE_citation_1": "Yushkevich, P., Zhang, G., & Developer, I. (2025, Aug 31).  [.]. Washington: NITRC.",
        "FILE_creator_1": "Yushkevich, P., Zhang, G., & Developer, I.",
        "FILE_publication_year_1": "2025",
        "FILE_version_1":"",
        "FILE_licence_1": "GNU General Public License (GPL)",
        "other_processor_1":"",
        "url_line_1_exists": "true",
        "url_processor_id_1": url_platform,
        "url_location_1": url_target,
        "url_name_1": url_name,
        "number_of_volumes_1":"",
        "url_doi_1":"",
        "URL_title_1": url_name,
        "URL_description_1":"",
        "URL_citation_1": "Yushkevich, P., Zhang, G., & Developer, I. (2025, Aug 31).  [.]. Washington: NITRC.",
        "URL_creator_1": "Yushkevich, P., Zhang, G., & Developer, I.",
        "URL_publication_year_1": "2025",
        "URL_version_1":"",
        "URL_licence_1": "GNU General Public License (GPL)",
        "url_other_processor_1":"",
        "submit": "Add Files",
    }
    r = session.post(f'https://www.nitrc.org/frs/admin/editrelease.php?group_id=110&release_id={release_id}&package_id={package_id}', data=payload)
    return r

def get_url_mapping(r):
    bs = BeautifulSoup(r.content)
    result = {}
    for form in bs.find_all('form', action=re.compile('editrelease.php')):
        sel = form.find('select', id='processor_id')
        if sel:
            file_id, url_name = None, None
            for input in form.find_all('input'):
                if input['name'] == 'file_id':
                    file_id = input['value']
                elif input['name'] == 'url_name':
                    url_name = input['value']
            if file_id and url_name:
                result[file_id] = url_name
                
    return result

In [8]:
def itksnap_add_release(session, version, package_id=2295):
    match = re.match('(.*)-([0-9]{8})$', version)
    if match is None:
        raise f'Invalid version string {version}'
    v_short, v_date = match[1], match[2]

    # Generate OS mappings (linux, windows, osx ARM, osx x86_64)
    sf_prefix = 'https://sourceforge.net/projects/itk-snap/files/itk-snap'
    platforms = [
        ('1001', 'Linux 64bit', f'{sf_prefix}/{v_short}/itksnap-{version}-Linux-x86_64.tar.gz'),
        ('1002', 'Windows 64bit', f'{sf_prefix}/{v_short}/itksnap-{version}-win64-AMD64.exe'),
        ('1003', 'MacOS x86_64', f'{sf_prefix}/{v_short}/itksnap-{version}-Darwin-x86_64.dmg'),
        ('1003', 'MacOS arm64', f'{sf_prefix}/{v_short}/itksnap-{version}-Darwin-arm64.dmg')]

    # Create or update the releases
    release_id = None
    for p in platforms:
        if release_id is None:    
            r = new_release(session, package_id=package_id, release_name=v_short, 
                            url_target=p[2], url_name=p[1], url_platform=p[0])
            release_id = parse_release_id(r)
            print(f'Generated new release {release_id}')
        else:
            r = add_link_to_release(session, package_id=package_id, release_name=v_short, 
                                    release_id=release_id, url_target=p[2], url_name=p[1], url_platform=p[0])

    # Get the created url id mappings
    return release_id, get_url_mapping(r)

In [9]:
# itksnap_add_release(session, '4.4.0-beta2-20250820')
itksnap_add_release(session, '4.4.0-20250909', package_id=127)

Generated new release 8162


(8162,
 {'20321': 'MacOS arm64',
  '20320': 'MacOS x86_64',
  '20319': 'Windows 64bit',
  '20318': 'Linux 64bit'})